# 🧠 CNN Architectures
**Course:** Deep Learning  
**Lecture:** 3 - Convolutional Neural Networks (CNNs)  
**Part:** c - CNN Architectures

---

> **Where we are:** We know the building blocks (conv, pool). Now we stack them into famous architectural patterns.

**What you'll learn:**
- LeNet style (shallow, large kernels)
- VGG style (deep, stacked 3x3 kernels)
- ResNet style (skip connections)

**Exam relevance:** ⭐⭐⭐ (Crucial to know why modern networks look the way they do.)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import time
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input, Activation, Add, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import cifar10

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 📖 Theory: Evolution of Architectures
1. **LeNet (1998):** Basic alternating Conv and Pool, followed by Dense layers.
2. **VGG (2014):** Realized that two $3 \times 3$ convolutions have the same receptive field as one $5 \times 5$, but with fewer parameters and an extra non-linearity. Stacked $3 \times 3$ convs everywhere.
3. **ResNet (2015):** As networks get deep, gradients vanish. Introduced **residual connections** (skip connections): $F(x) + x$. This allows gradients to flow directly through the skip path.

## 🔀 Your Options
At this step, you can choose between: LeNet-style, VGG-style, or ResNet-style.
| Option | Pros | Cons | When to use |
|---|---|---|---|
| LeNet-style | Very simple | Too shallow for hard tasks | Simple grayscale datasets (MNIST) |
| VGG-style | Easy to understand | Huge number of parameters | Rarely used as-is, but block pattern is common |
| ResNet-style | Can be extremely deep | Complex to code | Almost all modern applications |

In [ ]:
# ✏️ Your Turn: Implementing Blocks

def build_lenet():
    # ### YOUR CODE HERE ###
    # Implemenet Conv-Pool-Conv-Pool-Flatten-Dense-Dense
    pass

def build_vgg_block(x, filters):
    # ### YOUR CODE HERE ###
    # Add two Conv2D layers with 3x3 kernel, 'same' padding, relu activation
    # Followed by a MaxPooling2D layer
    pass

def build_residual_block(x, filters, stride=1):
    # ### YOUR CODE HERE ###
    # Branch 1: the skip connection. If stride != 1 or channels change, use a 1x1 conv
    # Branch 2: two Conv2D layers with 3x3 kernel, 'same' padding. 
    # Add Branch 1 and Branch 2, then Relu
    pass

In [ ]:
# ✅ Solution
def build_lenet():
    inp = Input(shape=(32, 32, 3))
    x = Conv2D(6, (5, 5), activation='relu')(inp)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(16, (5, 5), activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Flatten()(x)
    x = Dense(120, activation='relu')(x)
    x = Dense(84, activation='relu')(x)
    out = Dense(10, activation='softmax')(x)
    return Model(inp, out, name="LeNet_style")

def build_vgg_block(x, filters):
    x = Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    return x

def build_vgg_mini():
    inp = Input(shape=(32, 32, 3))
    x = build_vgg_block(inp, 32)
    x = build_vgg_block(x, 64)
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    out = Dense(10, activation='softmax')(x)
    return Model(inp, out, name="VGG_style")

def build_residual_block(x, filters, stride=1):
    shortcut = x
    
    # Match channels / downsample
    if stride != 1 or x.shape[-1] != filters:
        shortcut = Conv2D(filters, (1, 1), strides=stride, padding='same')(shortcut)
        
    x = Conv2D(filters, (3, 3), strides=stride, padding='same', activation='relu')(x)
    x = Conv2D(filters, (3, 3), padding='same')(x)
    
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def build_resnet_mini():
    inp = Input(shape=(32, 32, 3))
    x = Conv2D(32, (3, 3), padding='same', activation='relu')(inp)
    x = build_residual_block(x, 32)
    x = build_residual_block(x, 64, stride=2)
    x = GlobalAveragePooling2D()(x)
    out = Dense(10, activation='softmax')(x)
    return Model(inp, out, name="ResNet_style")

m1 = build_lenet()
m2 = build_vgg_mini()
m3 = build_resnet_mini()

In [ ]:
# 🔬 Comparison Cell: Architectures on CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
x_train, y_train = x_train[:10000], y_train[:10000]

stats = []
histories = {}

for m in [m1, m2, m3]:
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    params = m.count_params()
    
    start = time.time()
    h = m.fit(x_train, y_train, epochs=5, validation_split=0.2, verbose=0)
    time_per_epoch = (time.time() - start) / 5
    
    histories[m.name] = h.history
    stats.append({
        "Architecture": m.name,
        "# Params": params,
        "Train Acc": h.history['accuracy'][-1],
        "Val Acc": h.history['val_accuracy'][-1],
        "Time/epoch (s)": time_per_epoch
    })

import pandas as pd
display(pd.DataFrame(stats))

plt.figure(figsize=(10, 5))
for name, h in histories.items():
    plt.plot(h['val_accuracy'], label=f"{name} Val Acc")
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.title("Architecture Comparison")
plt.show()

## 🎓 Exam Corner

### Question 1 (Conceptual)
Why do modern networks prefer many 3×3 convolutions over single large kernels?

<details><summary>💡 Answer</summary>

Two stacked 3x3 convs have a receptive field of 5x5.
A single 5x5 with $C$ filters has $5 \times 5 \times C \times C = 25 C^2$ parameters.
Two 3x3s have $3 \times 3 \times C \times C \times 2 = 18 C^2$ parameters.
Fewer parameters, less computation, AND we get to insert an extra ReLU non-linearity in between, increasing expressive power.

</details>

### Question 2 (Conceptual)
What does a residual/skip connection solve? Write the formula.

<details><summary>💡 Answer</summary>

Formula: $output = ReLU(F(x) + x)$
It solves the vanishing gradient problem in very deep networks. During backpropagation, the gradient flows directly backwards through the addition operator (the $+x$ path), bypassing the potentially saturating convolutions.

</details>

### Question 3 (Conceptual)
Two stacked 3×3 convolutions have the same receptive field as a single convolution of what size?

<details><summary>💡 Answer</summary>

5x5.

</details>